## Notebook conventions

**Edit this file in place — don't save a new copy** (no `_V1`/`_V2`/dated/`-Copy`/`DEBUG-` variants). Commit changes via a branch + PR; git history is the version record, not the filename. Full conventions and `nbstripout` setup: see the repo [README](../../README.md) → "Notebook conventions."

Purpose of this notebook: to take apart video feed and turn each frame into an image that then can be run in MegaDetector, in batches.

Designed to run on JupyterHub / Linux / macOS / Windows
NOTE! this version of megadetector does not import a module, you install from a git clone into your directory and env with python 3.10, no higher.
install the requirements.txt inside the kernel/env you will create. ideas- frame rate function to detect? what files to keep? where? naming? use OOP? permissions for
users? how to manage files? recursive through subfolders? delete option for jsons and/or jpegs? what is workflow? what does timelapse need as input?
note if using chatgpt-must must stipulate which megadetector you are using, mixing pytorchwildlife and dan morris' would be disaster code. cuda and GPU being used?
how long does it take on a tower? Jetstream2? CPU only machine? test the same video on each system, run %timeit and compare.

Use the **"Python (MegaDetector 10)"** kernel (`megadetector10`) -- MegaDetector 10.0.24 is
installed system-wide at `/opt/tljh/user`, available to any user, no per-user conda env or
git clone needed. (An older, per-user `megadetector` conda env/kernel + git-clone setup used
to exist in `jupyter-bernie`'s home; that's been removed -- this kernel replaces it.)

Note: this has not been battle-tested for how many videos it can process!

In [ ]:
# Dan Morris' Megadetector version
# # MegaDetector v5 – video → frames → detection → video reconstruction  
# This notebook is now **platform independent** (Linux / macOS / Windows).
import os
import sys
import json
import warnings
import subprocess
import shutil
from pathlib import Path


import cv2                # opencv‑python‑headless works on all OSes
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output


import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {DEVICE}")

In [ ]:
# Replace with your own path or whatever -- this is set to the shared
# Barron Creek baseline dataset for the current baseline run.
#Example Folder containing videos (can have nested subfolders)
DATA_DIR = Path("/home/jupyter-bernie/Barron_Creek_10Aug2026/DCIM/100_BTCF")

# Output location (mirrors input structure)
# Baseline run 1 of 3 -- use a fresh empty folder per run (baseline_run1/2/3)
# so runs don't overwrite each other, matching the Tarazed-side protocol.
OUTPUT_DIR = Path("./megadetector_outputs/baseline_run1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model file: a real path to the weights, not a short name like "MDV5A" --
# a short name makes MegaDetector auto-download to $TMPDIR/megadetector_models,
# which is ephemeral /tmp on this server (can be wiped on reboot/cleanup,
# forcing a silent ~280MB re-download mid-run). Staged persistently instead
# at /shared/megadetector_models, in the same shared, multi-user location
# other projects on this server already use -- any user can read it here.
MODEL_NAME = "/shared/megadetector_models/md_v5a.0.1.pt"

# Frames per second for stitched video
OUTPUT_FPS = 30  # iPhone / camera trap safe default

# MegaDetector animal category ID (INT, not string)
ANIMAL_CATEGORY_ID = 1

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

print("Configuration:")
print(f"Input video folder: {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Model: {MODEL_NAME}")


In [ ]:
# Optional: start a timer so the summary cell at the end of this notebook can
# report elapsed wall-clock time. Safe to skip -- if you don't run this cell,
# the summary cell just skips the timing line instead of erroring.
import time
RUN_START = time.perf_counter()
print(f"Timer started at {time.strftime('%H:%M:%S')}")

In [ ]:
videos = [
    p for p in DATA_DIR.rglob("*")
    if p.suffix.lower() in VIDEO_EXTS
]

print(f"Found {len(videos)} videos")
videos[:5] #change the 5 to the number of videos if you would like to see them all and rerun cell


In [ ]:
# sanity check
import importlib

assert DATA_DIR.exists(), "Input data directory not found"
assert Path(MODEL_NAME).exists(), f"Model file not found: {MODEL_NAME}"
assert importlib.util.find_spec("megadetector") is not None, (
    "megadetector package not importable -- wrong kernel? "
    "This notebook expects the 'Python (MegaDetector 10)' kernel."
)

print("✔ All required paths exist")


In [ ]:
# Helper: find all videos recursively- important to make sure this is finding all videos in subfolders!
def find_all_videos(root_dir):
    VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}

    videos = []
    for root, _, files in os.walk(root_dir):
        root = Path(root)

        # Skip output directories
        if "megadetector_outputs" in root.parts:
            continue

        for f in files:
            p = Path(f)
            if (
                p.suffix.lower() in VIDEO_EXTS
                and "_detected" not in p.stem
            ):
                videos.append(root / p)

    return sorted(videos)



In [ ]:
# this really checks if you found all videos in subfolders or anywhere by calling function above
videos = find_all_videos(DATA_DIR)

print(f"Found {len(videos)} videos")
for v in videos[:10]:
    print(" ", v.relative_to(DATA_DIR))


In [ ]:
#checking what extensions exist in DATA_DIR- troubleshoots above if having issues
from collections import Counter

ext_counts = Counter()

for root, _, files in os.walk(DATA_DIR):
    for f in files:
        ext_counts[Path(f).suffix.lower()] += 1

ext_counts


In [ ]:
# Given a video path, where do its frames, JSON and output video live?
def video_output_dirs(video_path):
    """
    Given a video path under DATA_DIR, return all output paths
    """
    rel = video_path.relative_to(DATA_DIR)
    base = OUTPUT_DIR / rel.with_suffix("")

    frames_dir = base / "frames"
    json_path = base / "detections.json"
    output_video = base.with_name(base.name + "_detected.mp4")

    frames_dir.mkdir(parents=True, exist_ok=True)
    base.mkdir(parents=True, exist_ok=True)

    return frames_dir, json_path, output_video


In [ ]:
#troubleshooting issues
for video_path in videos:
    frames_dir, output_json, output_video = video_output_dirs(video_path)

    print("frames_dir =", frames_dir)
    print("OUTPUT_DIR =", OUTPUT_DIR)

    break  # just inspect the first one


In [ ]:
#helper clean frames directory safely
def cleanup_extractions(frames_dir):
    frames_dir = Path(frames_dir)

    # Safety checks
    assert frames_dir.name == "frames", f"Refusing to delete non-frames dir: {frames_dir}"
    assert OUTPUT_DIR in frames_dir.parents, f"{frames_dir} is not inside OUTPUT_DIR"

    if frames_dir.exists():
        shutil.rmtree(frames_dir)

    frames_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def resolve_frame_path(frames_dir, file_field):
    p = Path(file_field)

    # If MegaDetector already stored a path, use it
    if p.is_absolute() or p.exists():
        return p

    # Otherwise, assume it's relative to frames_dir
    return frames_dir / p


In [ ]:
def extract_frames(video_path, frames_dir):
    frames_dir = Path(frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "ffmpeg",
        "-y",                    # overwrite if exists
        "-i", str(video_path),
        "-q:v", "2",             # good JPEG quality
        str(frames_dir / "frame_%06d.jpg")
    ]

    subprocess.run(cmd, check=True)


In [ ]:
# helper check if frame has animal detections  HERE IS WHERE CONF THRESH IS SET TO .5
def has_animal_detection(frame_info, conf_thresh=0.5):
    for d in frame_info:
        if (
            d.get("category") == ANIMAL_CATEGORY_ID
            and d.get("confidence", 0) >= conf_thresh
        ):
            return True
    return False


In [ ]:
# helper draw bounding boxes (animal only)
def draw_boxes(frame, detections, conf_thresh=0.8):
    h, w = frame.shape[:2]

    for det in detections:
        if (
            det.get("category") != ANIMAL_CATEGORY_ID
            or det.get("confidence", 0) < conf_thresh
        ):
            continue

        x, y, bw, bh = det["bbox"]
        x1 = int(x * w)
        y1 = int(y * h)
        x2 = int((x + bw) * w)
        y2 = int((y + bh) * h)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"animal {det['confidence']:.2f}",
            (x1, max(y1 - 5, 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
        )

    return frame


In [ ]:
# helper stitch only frames with detections into video
def create_detected_video_only(md_json_path, frames_dir, output_video_path, conf_thresh=0.5):
    with open(md_json_path, "r") as f:
        md = json.load(f)

    images = md.get("images", [])

    # ---- Phase 1: global check ----
    has_any_animals = any(
        has_animal_detection(img.get("detections", []), conf_thresh)
        for img in images
    )

    if not has_any_animals:
        print("No animal detections found in entire video, skipping.")
        return

    # ---- Phase 2: initialize video writer using first readable frame ----
    first_frame = None
    first_path = None

    for img in images:
        p = frames_dir / img["file"]
        frame = cv2.imread(str(p))
        if frame is not None:
            first_frame = frame
            first_path = p
            break

    if first_frame is None:
        raise RuntimeError("No readable frames found to initialize video.")

    h, w = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_video_path), fourcc, OUTPUT_FPS, (w, h))

    written = 0

    # ---- Phase 3: write ALL frames ----
    for img in images:
        frame_path = frames_dir / img["file"]
        frame = cv2.imread(str(frame_path))
        if frame is None:
            continue

        detections = img.get("detections", [])

        # Draw boxes ONLY if animal detections exist
        if has_animal_detection(detections, conf_thresh):
            frame = draw_boxes(frame, detections, conf_thresh)

        out.write(frame)
        written += 1

    out.release()
    print(f"🎬 Wrote full video ({written} frames) to {output_video_path}")



In [ ]:
all_videos = find_all_videos(DATA_DIR)
print(f"Found {len(all_videos)} videos")

for video_path in all_videos:
    print("\n==============================")
    print("Processing:", video_path)

    #  ALWAYS derive paths this way
    frames_dir, output_json, output_video = video_output_dirs(video_path)
    #TESTS
    print("frames_dir:", frames_dir)
    print("output_json:", output_json)
    print("output_video:", output_video)
    # HARD RESET 
    cleanup_extractions(frames_dir)

    #EXTRACT
    extract_frames(video_path, frames_dir)

    # ASSERT: frames were actually extracted
    frame_files = sorted(frames_dir.glob("frame_*.jpg"))
    assert len(frame_files) > 0, f"No frames extracted for {video_path}"

    print(f"Extracted {len(frame_files)} frames")

    # --- Detect FPS ---
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    print(f"FPS: {fps}")

    # --- Extract frames ---
    cap = cv2.VideoCapture(str(video_path))
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_file = frames_dir / f"frame_{frame_count:05d}.jpg"
        cv2.imwrite(str(frame_file), frame)
        frame_count += 1
    cap.release()
    print(f"Extracted {frame_count} frames")

    # --- Run MegaDetector ---
    # Module invocation against the system-wide megadetector package (any user,
    # no repo clone / MEGADETECTOR_ROOT needed) instead of a hardcoded script path.
    !python -m megadetector.detection.run_detector_batch \
        {MODEL_NAME} \
        {frames_dir} \
        {output_json}

    # --- Load JSON ---
    with open(output_json) as f:
        md_json = json.load(f)

    image_detections = md_json.get("images", [])
    if not image_detections:
        print("⚠️ No detections, skipping video")
        cleanup_extractions(frames_dir)
        continue

    # --- Initialize video writer ---
    
    first_frame_path = resolve_frame_path(frames_dir, image_detections[0]["file"])
    first_frame = cv2.imread(str(first_frame_path))

    if first_frame is None:
        raise RuntimeError(f"Failed to read first frame: {first_frame_path}")

    height, width, _ = first_frame.shape

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_video), fourcc, fps, (width, height))

    # --- Write FULL video (boxes only on animal frames) ---
    written = 0
    for frame_info in image_detections:
        frame_path = resolve_frame_path(frames_dir, frame_info["file"])
        frame = cv2.imread(str(frame_path))
        if frame is None:
            continue


        if has_animal_detection(frame_info.get("detections", [])):
            frame = draw_boxes(frame, frame_info["detections"])

        out.write(frame)
        written += 1

    out.release()
    print(f"✅ Wrote {written} frames to {output_video}")

    # --- Cleanup ---
    cleanup_extractions(frames_dir)


In [ ]:

video_path = videos[0]  # debug with one video

frames_dir, output_json, output_video = video_output_dirs(video_path)

cleanup_extractions(frames_dir)

extract_frames(video_path, frames_dir)

run_megadetector(frames_dir, output_json)

create_detected_video_only(
    output_json,
    frames_dir,
    output_video
)

cleanup_extractions(frames_dir)


In [ ]:
with open(output_json) as f:
    md = json.load(f)

for img in md["images"]:
    if img.get("detections"):
        print(img["detections"][0])
        break


## Optional: Performance & Results Summary

Scans `OUTPUT_DIR` directly rather than depending on the main loop's in-memory state -- so this
works after any run, including a partial one, or after you've modified the loop yourself to test
an improvement. Run it any time after a batch (or part of one) has finished, to get comparable
numbers for before/after tracking.

In [ ]:
import time
import json
from pathlib import Path

detections_files = sorted(OUTPUT_DIR.rglob("detections.json"))
detected_videos = sorted(OUTPUT_DIR.rglob("*_detected.mp4"))

total_attempted = len(detections_files)
total_with_output = len(detected_videos)
total_skipped = total_attempted - total_with_output

total_frames = 0
unreadable = []
for f in detections_files:
    try:
        with open(f) as fh:
            md = json.load(fh)
        total_frames += len(md.get("images", []))
    except Exception as e:
        unreadable.append((f, e))

print("=" * 50)
print("RUN SUMMARY")
print("=" * 50)
print(f"Videos attempted (detections.json found)   : {total_attempted}")
print(f"Videos with output video (had detections)  : {total_with_output}")
print(f"Videos skipped (no detections)              : {total_skipped}")
print(f"Total frames processed by MegaDetector      : {total_frames}")

if unreadable:
    print(f"\n⚠️  {len(unreadable)} detections.json file(s) couldn't be read:")
    for f, e in unreadable:
        print(f"   {f}: {e}")

if "RUN_START" in globals():
    elapsed = time.perf_counter() - RUN_START
    print(f"\nElapsed wall-clock time                     : {elapsed:.1f}s ({elapsed/60:.1f} min)")
    if total_attempted:
        print(f"Average time per video                      : {elapsed/total_attempted:.1f}s")
else:
    print("\nElapsed wall-clock time                     : not tracked")
    print("  (run the 'start a timer' cell near the top before your batch run to get this)")
print("=" * 50)